# Training Model



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '3qidigmfd8aaav'
os.environ['DataZoneDomainId'] = 'dzd-d0lbnojk3v4487'
os.environ['DataZoneEnvironmentId'] = '5prh9p9eqrfgo7'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "3qidigmfd8aaav",
                "DataZoneDomainId": "dzd-d0lbnojk3v4487",
                "DataZoneEnvironmentId": "5prh9p9eqrfgo7",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

# AWS Fraud Detection Analysis
# SageMaker XGBoost Training Notebook
#
# Goal:
# 1. Read fraud transaction dataset from S3
# 2. Preprocess categorical fields
# 3. Train XGBoost fraud detection model
# 4. Save model artifact to S3 model/ folder
# 5. Deploy endpoint for Lambda real-time inference

In [0]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sagemaker.inputs import TrainingInput
from sagemaker.image_uris import retrieve
from sagemaker.estimator import Estimator
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

In [0]:
region = boto3.Session().region_name
session = sagemaker.Session()
role = sagemaker.get_execution_role()

bucket = "finalproject-fraud-detection"
raw_prefix = "raw"
processed_prefix = "processed"
model_prefix = "model"

input_s3_uri = f"s3://{bucket}/{raw_prefix}/fraud_transactions.csv"
train_s3_uri = f"s3://{bucket}/{processed_prefix}/train/train.csv"
validation_s3_uri = f"s3://{bucket}/{processed_prefix}/validation/validation.csv"
output_path = f"s3://{bucket}/{model_prefix}/"

print("Region:", region)
print("Bucket:", bucket)
print("Input:", input_s3_uri)
print("Model Output:", output_path)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Region: us-east-1
Bucket: finalproject-fraud-detection
Input: s3://finalproject-fraud-detection/raw/fraud_transactions.csv
Model Output: s3://finalproject-fraud-detection/model/


In [0]:
df = pd.read_csv(input_s3_uri)
df.head()

print(df.shape)
print(df.columns.tolist())
print(df["fraud"].value_counts())
print(df.isnull().sum())

(1000, 7)
['transaction_id', 'timestamp', 'amount', 'merchant', 'location', 'payment_method', 'fraud']
fraud
0    900
1    100
Name: count, dtype: int64
transaction_id    0
timestamp         0
amount            0
merchant          0
location          0
payment_method    0
fraud             0
dtype: int64


In [0]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month

merchant_map = {
    "Amazon": 0,
    "Walmart": 1,
    "Apple Store": 2,
    "Best Buy": 3,
    "Target": 4,
    "Starbucks": 5,
    "eBay": 6,
    "Uber": 7,
    "Netflix": 8
}

location_map = {
    "New York": 0,
    "Boston": 1,
    "San Francisco": 2,
    "Chicago": 3,
    "Los Angeles": 4,
    "Seattle": 5,
    "Dallas": 6,
    "Miami": 7
}

payment_map = {
    "Credit Card": 0,
    "Debit Card": 1,
    "Online Payment": 2,
    "Mobile Pay": 3
}

df["merchant_enc"] = df["merchant"].map(merchant_map)
df["location_enc"] = df["location"].map(location_map)
df["payment_method_enc"] = df["payment_method"].map(payment_map)

In [0]:
feature_columns = [
    "amount",
    "hour",
    "day",
    "month",
    "merchant_enc",
    "location_enc",
    "payment_method_enc"
]

target_column = "fraud"

X = df[feature_columns]
y = df[target_column]

print("Features:")
print(X.head())

Features:
    amount  hour  day  month  merchant_enc  location_enc  payment_method_enc
0   404.43     0    1      8             3             5                   3
1   587.06     0    9      5             2             0                   0
2  1972.41     0   13      3             5             3                   0
3   445.41     0   10      7             2             7                   1
4  1704.13     0    8     11             8             5                   1


In [0]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = pd.concat([y_train, X_train], axis=1)
val_df = pd.concat([y_val, X_val], axis=1)

print(train_df.shape, val_df.shape)

(800, 8) (200, 8)


In [0]:
train_df.to_csv("train.csv", header=False, index=False)
val_df.to_csv("validation.csv", header=False, index=False)

session.upload_data("train.csv", bucket=bucket, key_prefix=f"{processed_prefix}/train")
session.upload_data("validation.csv", bucket=bucket, key_prefix=f"{processed_prefix}/validation")

print("Uploaded training data to:", train_s3_uri)
print("Uploaded validation data to:", validation_s3_uri)

Uploaded training data to: s3://finalproject-fraud-detection/processed/train/train.csv
Uploaded validation data to: s3://finalproject-fraud-detection/processed/validation/validation.csv


In [0]:
container = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1"
)

print(container)

683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [0]:
xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_path,
    sagemaker_session=session
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    num_round=100,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=1
)

In [0]:
train_input = TrainingInput(
    s3_data=train_s3_uri,
    content_type="text/csv"
)

validation_input = TrainingInput(
    s3_data=validation_s3_uri,
    content_type="text/csv"
)

In [0]:
xgb.fit({
    "train": train_input,
    "validation": validation_input
})

2026-03-10 19:35:32 Starting - Starting the training job.

.

.


2026-03-10 19:35:57 Starting - Preparing the instances for training.

.

...

.

.


2026-03-10 19:38:31 Training - Training image download completed. Training in progress.
2026-03-10 19:38:31 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-10 19:38:25.409 ip-10-0-72-241.ec2.internal:8 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-10 19:38:25.496 ip-10-0-72-241.ec2.internal:8 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-10:19:38:25:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-10:19:38:25:INFO] Failed to parse hyperparameter objective value binary:logistic to Json.
Returning the value itself
[2026-03-10:19:38:25:INFO] No GPUs detected (normal if no gpus inst


2026-03-10 19:38:45 Completed - Training job completed


Training seconds: 145
Billable seconds: 145


In [0]:
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

endpoint_name = predictor.endpoint_name
print("Endpoint Name:", endpoint_name)

-

-

-

-

-

-

!

Endpoint Name: sagemaker-xgboost-2026-03-10-19-39-20-161


In [0]:
sample = X_val.iloc[0].tolist()
payload = ",".join(map(str, sample))

result = predictor.predict(payload)
print(result)

{'predictions': [{'score': 0.09094083309173584}]}


In [0]:
metadata = {
    "bucket": bucket,
    "model_output_path": output_path,
    "endpoint_name": endpoint_name,
    "feature_columns": feature_columns
}

print(metadata)

{'bucket': 'finalproject-fraud-detection', 'model_output_path': 's3://finalproject-fraud-detection/model/', 'endpoint_name': 'sagemaker-xgboost-2026-03-10-19-39-20-161', 'feature_columns': ['amount', 'hour', 'day', 'month', 'merchant_enc', 'location_enc', 'payment_method_enc']}


## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()